# Token Classification (NER): Fine-tuning BERT

A simple, end-to-end example of fine-tuning BERT for **Named Entity Recognition** — labeling each token in a sentence (person, organization, location, etc.).

Built with `transformers` + `torch`.

## 0. Install dependencies

Run once if these aren't already installed.

In [ ]:
# !pip install torch transformers datasets seqeval pandas

In [ ]:
!pip install seqeval
## used for evaluating token classifications

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=c6f686503aa51f85ae1e08da94f721d84857149b6850f7e78c6d433308fa2158
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports

In [ ]:
import json

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from seqeval.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## 2. Download a public dataset and 3. save it to CSV

We use **CoNLL-2003**. Each example is a list of `tokens` and a matching list of `ner_tags` (integer label per token).

Because CSV cells are flat, we serialize the token/tag lists as JSON strings — one sentence per row. To keep training fast, we take a small subset.

In [ ]:
# # Download the raw JSON-lines data files directly with pandas (no datasets loader / no script)
# BASE = "https://huggingface.co/datasets/tner/conll2003/resolve/main/dataset"

# # Label mapping ships as a small JSON object: {"O": 0, "B-ORG": 1, ...}
# label2id_src = pd.read_json(f"{BASE}/label.json", typ="series").to_dict()
# label_names = [None] * len(label2id_src)
# for name, i in label2id_src.items():
#     label_names[i] = name
# print("Labels:", label_names)

# # Each line is {"tokens": [...], "tags": [...]}
# train_full = pd.read_json(f"{BASE}/train.json", lines=True)
# valid_full = pd.read_json(f"{BASE}/valid.json", lines=True)

# # Take a small subset so it trains quickly (use the full frames for best accuracy)
# train_subset = train_full.sample(n=2000, random_state=42).reset_index(drop=True)
# test_subset = valid_full.sample(n=500, random_state=42).reset_index(drop=True)


# def to_csv(df, path):
#     # Save token/tag lists as JSON strings; store "tags" under the "ner_tags" column
#     out = pd.DataFrame(
#         {
#             "tokens": df["tokens"].apply(json.dumps),
#             "ner_tags": df["tags"].apply(json.dumps),
#         }
#     )
#     out.to_csv(path, index=False)



# to_csv(train_subset, "/content/drive/MyDrive/MLDatasets/token_classification/train.csv")
# to_csv(test_subset, "/content/drive/MyDrive/MLDatasets/token_classification/test.csv")
# print("Saved train.csv and test.csv")

Labels: ['O', 'B-ORG', 'B-MISC', 'B-PER', 'I-PER', 'B-LOC', 'I-ORG', 'I-MISC', 'I-LOC']
Saved train.csv and test.csv


## 4. Read the data back from the CSV files

We `json.loads` the two columns back into Python lists.

In [ ]:
def read_csv(path):
    df = pd.read_csv(path)
    df["tokens"] = df["tokens"].apply(json.loads)
    df["ner_tags"] = df["ner_tags"].apply(json.loads)
    return df


train_df = read_csv("/content/drive/MyDrive/MLDatasets/token_classification/train.csv")
test_df = read_csv("/content/drive/MyDrive/MLDatasets/token_classification/test.csv")

label_names = ['O', 'B-ORG', 'B-MISC', 'B-PER', 'I-PER', 'B-LOC', 'I-ORG', 'I-MISC', 'I-LOC']

num_labels = len(label_names)
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

print(train_df.shape, test_df.shape)
print(train_df["tokens"].iloc[0])
print(train_df["ner_tags"].iloc[0])

(2000, 2) (500, 2)
['Attendance', ':', '1,800']
[0, 0, 0]


In [ ]:
train_df

,tokens,ner_tags
0,"[Attendance, :, 1,800]","[0, 0, 0]"
1,"[Ties, between, the, two, neighbours, ,, strai...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, ..."
2,"[The, winner, has, to, secure, a, majority, fr...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[LIBERIA, ,, Costa, Rica, 1996-08-26]","[5, 0, 5, 8, 0]"
4,"[1., Sorensen, 11.20:33]","[0, 3, 0]"
...,...,...
1995,"[The, Salang, tunnel, ,, the, main, supply, ro...","[0, 5, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, ..."
1996,"[The, opposition, party, Radical, Cause, contr...","[0, 0, 0, 1, 6, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
1997,"[Maturity, Balance, Coupon, List]","[0, 0, 0, 0]"
1998,"[Result, of, a, Scottish, premier]","[0, 0, 0, 2, 0]"


In [ ]:
'''
'O' - other

ORGANIZATION
'B-ORG'
'I-ORG'

PERSON
'B-PER'
'I-PER'

LOCATION
'B-LOC'
'I-LOC'


MISCELENIOUS
'B-MISC'
'I-MISC'




Elon Musk and X Musk are CEOs of SpaceX Inc Limited

Elon Musk -> PER
X Musk -> PER
SpaceX - ORG

Elon - B-PER
Musk - I-PER
and - O (other)
X - B-PER
Musk - I-PER
are - O
CEOs - O
of - O
SpaceX - B-ORG
Inc - I-ORG
Limited - I-ORG
'''

In [ ]:
label2id

{'O': 0,
 'B-ORG': 1,
 'B-MISC': 2,
 'B-PER': 3,
 'I-PER': 4,
 'B-LOC': 5,
 'I-ORG': 6,
 'I-MISC': 7,
 'I-LOC': 8}

In [ ]:
train_df

,tokens,ner_tags
0,"[Attendance, :, 1,800]","[0, 0, 0]"
1,"[Ties, between, the, two, neighbours, ,, strai...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, ..."
2,"[The, winner, has, to, secure, a, majority, fr...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[LIBERIA, ,, Costa, Rica, 1996-08-26]","[5, 0, 5, 8, 0]"
4,"[1., Sorensen, 11.20:33]","[0, 3, 0]"
...,...,...
1995,"[The, Salang, tunnel, ,, the, main, supply, ro...","[0, 5, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, ..."
1996,"[The, opposition, party, Radical, Cause, contr...","[0, 0, 0, 1, 6, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
1997,"[Maturity, Balance, Coupon, List]","[0, 0, 0, 0]"
1998,"[Result, of, a, Scottish, premier]","[0, 0, 0, 2, 0]"


## 5. Tokenize with label alignment

BERT splits words into subwords, so each word's label must be spread across its subword pieces. Convention: label the **first** subword of a word, and mark the rest (plus special tokens) with `-100` so they're ignored by the loss.

In [ ]:
MODEL_NAME = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def align_labels(tokens, tags):
    enc = tokenizer(
        tokens,
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )
    word_ids = enc.word_ids()
    labels = []
    prev = None
    for wid in word_ids:
        if wid is None:
            labels.append(-100)          # special tokens ([CLS], [SEP], pad)
        elif wid != prev:
            labels.append(tags[wid])     # first subword of a word
        else:
            labels.append(-100)          # continuation subwords
        prev = wid
    enc["labels"] = labels
    return enc


class NERDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.examples = [
            align_labels(t, l)
            for t, l in zip(df["tokens"], df["ner_tags"])
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


train_ds = NERDataset(train_df)
test_ds = NERDataset(test_df)

# Pads input_ids AND labels (with -100) to the longest sequence in each batch
data_collator = DataCollatorForTokenClassification(tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
train_df['tokens'][3]

['LIBERIA', ',', 'Costa', 'Rica', '1996-08-26']

In [ ]:
train_df['ner_tags'][3]

[5, 0, 5, 8, 0]

In [ ]:
i=4
tokens = train_df.iloc[i]['tokens']
tags = train_df.iloc[i]['ner_tags']
token_ids=align_labels(tokens,tags)
token_ids

{'input_ids': [101, 122, 119, 1573, 5123, 3792, 1429, 119, 1406, 131, 3081, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, -100, 3, -100, -100, 0, -100, -100, -100, -100, -100]}

In [ ]:
print(train_df.iloc[4].to_dict())

{'tokens': ['1.', 'Sorensen', '11.20:33'], 'ner_tags': [0, 3, 0]}


In [ ]:
label2id

{'O': 0,
 'B-ORG': 1,
 'B-MISC': 2,
 'B-PER': 3,
 'I-PER': 4,
 'B-LOC': 5,
 'I-ORG': 6,
 'I-MISC': 7,
 'I-LOC': 8}

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(token_ids['input_ids'])
for t,id,label in zip(tokens,token_ids['input_ids'],token_ids['labels']):
  print(f"{t} {id}  {label}")

[CLS] 101  -100
1 122  0
. 119  -100
So 1573  3
##ren 5123  -100
##sen 3792  -100
11 1429  0
. 119  -100
20 1406  -100
: 131  -100
33 3081  -100
[SEP] 102  -100


## 6. Fine-tune (training)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Drop the -100 positions and map ids back to label strings for seqeval
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        p, l = [], []
        for pr, la in zip(pred_row, label_row):
            if la != -100:
                p.append(label_names[pr])
                l.append(label_names[la])
        true_preds.append(p)
        true_labels.append(l)

    return {
        "accuracy": accuracy_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


training_args = TrainingArguments(
    output_dir="./ner-output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## to make training faster , increase batch sizes if GPU memeory is free (nvidia-smi in terminal shows GPU memory)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.128345,0.125482,0.966667,0.815252
2,0.083412,0.079263,0.979151,0.876344
3,0.047678,0.072189,0.979665,0.882101


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=375, training_loss=0.18103103669484458, metrics={'train_runtime': 87.3607, 'train_samples_per_second': 68.681, 'train_steps_per_second': 4.293, 'total_flos': 152623295375040.0, 'train_loss': 0.18103103669484458, 'epoch': 3.0})

## 7. Evaluate (per-entity report)

In [ ]:
print(trainer.evaluate())

# Detailed precision/recall/F1 per entity type
pred_output = trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

true_preds, true_labels = [], []
for pred_row, label_row in zip(preds, labels):
    p, l = [], []
    for pr, la in zip(pred_row, label_row):
        if la != -100:
            p.append(label_names[pr])
            l.append(label_names[la])
    true_preds.append(p)
    true_labels.append(l)

print(classification_report(true_labels, true_preds))

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.047678,0.072189,3,0.979665,0.882101


{'eval_loss': 0.0721888393163681, 'eval_accuracy': 0.9796653796653797, 'eval_f1': 0.8821007502679529}


              precision    recall  f1-score   support

         LOC       0.91      0.92      0.91       303
        MISC       0.74      0.76      0.75       128
         ORG       0.77      0.83      0.80       189
         PER       0.97      0.96      0.96       304

   micro avg       0.87      0.89      0.88       924
   macro avg       0.85      0.87      0.86       924
weighted avg       0.88      0.89      0.88       924



## 8. Save the fine-tuned model

In [ ]:
trainer.save_model("./bert-ner")
tokenizer.save_pretrained("./bert-ner")
print("Model saved to ./bert-ner")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bert-ner


## 9. Inference on new text

The `token-classification` pipeline handles tokenization and groups subwords back into whole entities.

In [ ]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # merge subwords into full entity spans
    device=0 if device == "cuda" else -1,
)

samples = [
    "Elon Musk founded SpaceX in California.",
    "Angela Merkel met the President of France in Berlin.",
    "Apple released the new iPhone in Cupertino last week.",
    "New Delhi is capital of India"
]

for text in samples:
    print(text)
    for ent in ner(text):
        print(f"  {ent['word']:<20} -> {ent['entity_group']} ({ent['score']:.2f})")
    print()

Elon Musk founded SpaceX in California.
  El                   -> PER (0.97)
  ##on Musk            -> PER (0.89)
  SpaceX               -> ORG (0.92)
  California           -> LOC (0.96)

Angela Merkel met the President of France in Berlin.
  Angela Merkel        -> PER (0.89)
  France               -> LOC (0.95)
  Berlin               -> LOC (0.98)

Apple released the new iPhone in Cupertino last week.
  Apple                -> ORG (0.96)
  Cupertino            -> LOC (0.80)

New Delhi is capital of India
  New Delhi            -> LOC (0.92)
  India                -> LOC (0.95)

